In [34]:
import pandas as pd
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

# nltk.download("stopwords")

texts = []
labels = []

with open("./resources/tweets_train.txt", "r", encoding="utf-8") as f:
    for line in f:
        label, tweet = line.strip().split(",", 1)
        labels.append(label)
        texts.append(tweet)

df = pd.DataFrame({
    "label": labels,
    "tweet": texts
})

# Prepro
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()
def prepro(text):

    text = text.lower()

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    tokens = nltk.word_tokenize(text)

    tokens = [
        word for word in tokens
        if word not in stop_words
    ]

    stemmer = PorterStemmer()

    stemmed_tokens = [
        stemmer.stem(word)
        for word in tokens
    ]

    return " ".join(stemmed_tokens)

df["clean_tweet"] = df["tweet"].apply(prepro)
# print(df["clean_tweet"].head())
vectorizer = CountVectorizer(max_features=500)
X_counts = vectorizer.fit_transform(df["clean_tweet"])
X_counts

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 37334 stored elements and shape (6588, 500)>

In [35]:
import pandas as pd

count_vectorized_df = pd.DataFrame.sparse.from_spmatrix(
    X_counts,
    columns=vectorizer.get_feature_names_out()
)
print(count_vectorized_df.iloc[:3,400-1:403-1].to_markdown())

|    |   someth |   son |   song |
|---:|---------:|------:|-------:|
|  0 |        0 |     0 |      0 |
|  1 |        0 |     0 |      0 |
|  2 |        0 |     0 |      0 |


In [36]:
print(
    count_vectorized_df.iloc[3][
        count_vectorized_df.iloc[3] > 0
    ]
)

cant    1
deal    1
end     1
find    1
keep    1
like    1
may     1
say     1
talk    1
Name: 3, dtype: Sparse[int64, 0]


In [37]:
word_counts = count_vectorized_df.sum(axis=0)

top15 = word_counts.sort_values(ascending=False).head(15)

print(top15)

tomorrow    1126
go           733
day          667
night        641
may          533
tonight      501
see          439
time         429
im           422
get          398
today        389
game         382
saturday     379
friday       375
sunday       368
dtype: Sparse[int64, 0]


In [38]:
label_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

count_vectorized_df["label"] = df["label"].map(label_map)
print(count_vectorized_df.iloc[350:354, 499:501].to_markdown())

|     |   your |   label |
|----:|-------:|--------:|
| 350 |      0 |       1 |
| 351 |      1 |      -1 |
| 352 |      0 |       1 |
| 353 |      0 |       0 |
